# C06 Weighted Sampler tuning

Train with city-group weighted sampling and plain cross-entropy to compare a simpler rebalancing baseline against loss-based challengers.


In [1]:
import json, random, numpy as np, pandas as pd, torch, joblib
from pathlib import Path
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "notebooks" / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"CWD: {CWD}")
print(f"NOTEBOOKS_DIR: {NOTEBOOKS_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / 'train.csv').exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")


CWD: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/challengers
NOTEBOOKS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks
PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
DATA_DIR exists: True
MODELS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers
FIGURES_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers


In [3]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts().sort_index()
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {len(city_counts)}")
print("City counts:", city_counts.to_dict())


Train: 16530, Val: 5510, Test: 5510
Labels: 9, City groups: 41
City counts: {'Other': 3707, 'Алматы': 192, 'Барнаул': 91, 'Владивосток': 80, 'Волгоград': 114, 'Воронеж': 214, 'Екатеринбург': 292, 'Ижевск': 81, 'Иркутск': 91, 'Казань': 353, 'Калининград': 81, 'Калуга': 71, 'Кемерово': 59, 'Краснодар': 432, 'Красноярск': 171, 'Липецк': 72, 'Минск': 128, 'Москва': 5617, 'Набережные Челны': 57, 'Нижний Новгород': 235, 'Новосибирск': 361, 'Омск': 124, 'Оренбург': 63, 'Пенза': 63, 'Пермь': 181, 'Ростов-на-Дону': 243, 'Самара': 271, 'Санкт-Петербург': 1797, 'Саратов': 99, 'Симферополь': 67, 'Сочи': 61, 'Ставрополь': 64, 'Тверь': 63, 'Тольятти': 77, 'Томск': 107, 'Тюмень': 150, 'Ульяновск': 71, 'Уфа': 222, 'Хабаровск': 76, 'Челябинск': 124, 'Ярославль': 108}


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train[["resume_text", "y"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Map: 100%|██████████| 5510/5510 [00:07<00:00, 731.98 examples/s]


In [5]:
def make_sampler_weights(groups, power):
    counts = groups.value_counts()
    raw = 1.0 / np.power(counts.astype(float), power)
    normalized = raw / raw.mean()
    weights = groups.map(normalized).astype(float).to_numpy()
    return weights, normalized.to_dict()


class WeightedSamplerTrainer(Trainer):
    def __init__(self, *args, sampler_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.sampler_weights = torch.as_tensor(sampler_weights, dtype=torch.double)
        print(f"Weighted sampler active: min={self.sampler_weights.min().item():.4f}, max={self.sampler_weights.max().item():.4f}")

    def get_train_dataloader(self):
        sampler = WeightedRandomSampler(
            weights=self.sampler_weights,
            num_samples=len(self.train_dataset),
            replacement=True,
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


In [6]:
RUNS = [
    {"tag": "citypow05_2ep", "power": 0.5, "epochs": 2},
    {"tag": "citypow10_2ep", "power": 1.0, "epochs": 2},
    {"tag": "citypow15_2ep", "power": 1.5, "epochs": 2},
]

summary_rows = []

for run in RUNS:
    model_name = f"weighted_sampler_{run['tag']}"
    save_dir = MODELS_DIR / model_name
    checkpoint_dir = save_dir / "checkpoints"
    sampler_weights, sampler_map = make_sampler_weights(df_train["city_group"], run["power"])

    print()
    print("=" * 80)
    print(f"Training {model_name}: power={run['power']}, epochs={run['epochs']}")

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=run["epochs"],
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )

    trainer = WeightedSamplerTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        sampler_weights=sampler_weights,
    )

    trainer.train()
    pred_output = trainer.predict(test_ds)
    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=1)

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    tpr, support = ovr_rates(df_eval, "city_group", num_labels)
    worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    joblib.dump(le, save_dir / "label_encoder.joblib")

    config = {
        "method": "Weighted Sampler (city_group) + Plain CE",
        "sampling_power": run["power"],
        "epochs": run["epochs"],
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "sampler_weights_by_city": sampler_map,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "tpr_gap_worst_robust": float(worst_gap),
        "tpr_gap_macro_robust": float(macro_gap),
    }
    with open(save_dir / "training_config.json", "w") as f:
        json.dump(config, f, indent=2)

    summary_rows.append({
        "model_name": model_name,
        "power": run["power"],
        "epochs": run["epochs"],
        "accuracy": acc,
        "macro_f1": macro_f1,
        "worst_gap": float(worst_gap),
        "macro_gap": float(macro_gap),
        "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["worst_gap", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
summary_path = FIGURES_DIR / "c06_weighted_sampler_tuning_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df



Training weighted_sampler_citypow05_2ep: power=0.5, epochs=2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 612.40it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Weighted sampler active: min=0.1481, max=1.4699


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.130987,1.138537,0.588748,0.598069
2,0.987609,1.143746,0.611071,0.618714


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

...

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.041182,1.388307,0.562432,0.567158
2,0.574025,1.476087,0.575318,0.573610


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.74s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.22s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]


,model_name,power,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,weighted_sampler_citypow15_2ep,1.5,2,0.560980,0.560925,0.288235,0.107185,notebooks/models/challengers/weighted_sampler_...
1,weighted_sampler_citypow10_2ep,1.0,2,0.577858,0.583044,0.329412,0.131475,notebooks/models/challengers/weighted_sampler_...
2,weighted_sampler_citypow05_2ep,0.5,2,0.591833,0.601606,0.352941,0.148083,notebooks/models/challengers/weighted_sampler_...


In [7]:
print(f"Summary saved to: {summary_path}")
summary_df


Summary saved to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers/c06_weighted_sampler_tuning_summary.csv


,model_name,power,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,weighted_sampler_citypow15_2ep,1.5,2,0.560980,0.560925,0.288235,0.107185,notebooks/models/challengers/weighted_sampler_...
1,weighted_sampler_citypow10_2ep,1.0,2,0.577858,0.583044,0.329412,0.131475,notebooks/models/challengers/weighted_sampler_...
2,weighted_sampler_citypow05_2ep,0.5,2,0.591833,0.601606,0.352941,0.148083,notebooks/models/challengers/weighted_sampler_...
